In [ ]:
!pip install langchain langchain-openai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.2 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install langchain langchain-google-genai langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.2 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -qU langchain langchain-google-genai langchain-community faiss-cpu langchain-text-splitters
print("✅ Dependencias instaladas")

✅ Dependencias instaladas


In [ ]:
import os
from google.colab import drive, userdata
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Montar Drive y configurar API
drive.mount('/content/drive')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# 2. Cargar el Vector Store de la ep1
ruta_faiss = "/content/drive/MyDrive/EP1/faiss_index_academico"
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = FAISS.load_local(ruta_faiss, embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente académico. Responde usando ÚNICAMENTE el CONTEXTO. Si no está, di 'INFORMACIÓN NO DISPONIBLE'."),
    ("human", "CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}")
])

def formatear_contexto(docs):
    return "\n\n---\n\n".join(d.page_content for d in docs)

cadena_rag = (
    {"contexto": retriever | formatear_contexto, "pregunta": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)
print("✅ RAG de EP1 cargado y listo para usarse")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ RAG de EP1 cargado y listo para usarse


In [ ]:
from langchain_core.tools import tool

# 1. HERRAMIENTA DE CONSULTA
@tool
def buscar_reglamento_educativo(pregunta: str) -> str:
    """Busca normativas, mallas curriculares y reglas en los documentos oficiales de la institución."""
    # Invoca la cadena RAG que creamos en la celda anterior
    respuesta = cadena_rag.invoke(pregunta)
    return respuesta

# 2. HERRAMIENTA DE RAZONAMIENTO (IE5 - Toma de decisiones)
@tool
def validar_prerrequisito(notas_alumno: str, asignatura: str) -> str:
    """Verifica si el alumno cumple con los prerrequisitos para una asignatura."""
    # Lógica simulada para la toma de decisiones
    if "Base de Datos I" not in notas_alumno and asignatura == "Base de Datos II":
        return "Resultado: RECHAZADO. El alumno no tiene aprobada 'Base de Datos I'."
    return "Resultado: APROBADO. Cumple los prerrequisitos."

# 3. HERRAMIENTA DE ESCRITURA (IE1)
@tool
def redactar_correo(destinatario: str, asunto: str, mensaje: str) -> str:
    """Redacta un correo formal para el estudiante o la coordinación académica."""
    return f"--- BORRADOR DE CORREO ---\nPara: {destinatario}\nAsunto: {asunto}\n\n{mensaje}\n--------------------------"

tools = [buscar_reglamento_educativo, validar_prerrequisito, redactar_correo]
print("✅ Herramientas creadas")

✅ Herramientas creadas


In [24]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.memory import ConversationBufferMemory
import os

# 1. MEMORIA A CORTO PLAZO (Historial de la conversación actual)
memory_corto_plazo = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    input_key="input"
)

# 2. MEMORIA A LARGO PLAZO (Persistencia en archivo)
ruta_memoria_largo_plazo = "/content/drive/MyDrive/EP1/memoria_largo_plazo.txt"

def guardar_en_memoria_largo_plazo(texto):
    with open(ruta_memoria_largo_plazo, "a", encoding="utf-8") as f:
        f.write(texto + "\n")

def leer_memoria_largo_plazo():
    if os.path.exists(ruta_memoria_largo_plazo):
        with open(ruta_memoria_largo_plazo, "r", encoding="utf-8") as f:
            return f.read()
    return "No hay historial previo."

# PROMPT DE ORQUESTACIÓN (IE5)
prompt_agente = ChatPromptTemplate.from_messages([
    ("system", """Eres un Asesor Académico Virtual.
    Historial de interacciones pasadas (Memoria a Largo Plazo):
    {memoria_largo_plazo}

    Tu proceso de planificación paso a paso es:
    1. Analiza la consulta del alumno.
    2. Usa 'buscar_reglamento_educativo' para consultar la base de datos oficial.
    3. Si el alumno pide inscribir una asignatura, usa 'validar_prerrequisito'.
    4. Si la validación es RECHAZADA, usa 'redactar_correo' para generar una solicitud de excepción.
    Responde de forma clara y profesional."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Creando Agente y Ejecutor
agente = create_tool_calling_agent(llm, tools, prompt_agente)

agent_executor = AgentExecutor(
    agent=agente,
    tools=tools,
    memory=memory_corto_plazo,
    verbose=True
)
print("✅ Agente Autónomo con Memoria de Corto y Largo Plazo listo")

✅ Agente Autónomo con Memoria de Corto y Largo Plazo listo


In [25]:
# ep1: Consulta al RAG Oficial
print("--- PRUEBA 1: Consulta al RAG Oficial ---")
resp1 = agent_executor.invoke({
    "input": "¿Cuál es el requisito de asistencia para aprobar un ramo?",
    "memoria_largo_plazo": leer_memoria_largo_plazo()
})
print("\n🤖 Agente responde:", resp1['output'])

print("\n" + "="*50 + "\n")

# PRUEBA 2: Decisión adaptativa y Escritura (IE5 e IE7)
print("--- PRUEBA 2: Decisión y Escritura ---")
resp2 = agent_executor.invoke({
    "input": "Quiero inscribir Base de Datos II, pero mis notas solo dicen que aprobé Programación. Redacta un correo a la coordinadora (coordinacion@duoc.cl) para pedir una excepción.",
    "memoria_largo_plazo": leer_memoria_largo_plazo()
})
print("\n🤖 Agente responde:", resp2['output'])

print("\n" + "="*50 + "\n")

# Demostración de Guardado en Memoria a Largo Plazo
# Guardar una interacción en la memoria de largo plazo
guardar_en_memoria_largo_plazo("El alumno Juan Pérez consultó por inscripción de Base de Datos II el día de hoy.")
print("✅ Interacción guardada en memoria de largo plazo.")

--- PRUEBA 1: Consulta al RAG Oficial ---


> Entering new AgentExecutor chain...

Invoking: `buscar_reglamento_educativo` with `{'pregunta': '¿Cuál es el requisito de asistencia para aprobar un ramo?'}`


Para aprobar cualquier asignatura, el estudiante debe cumplir con un mínimo de 75% de asistencia a clases teóricas y 100% a clases prácticas obligatorias.[{'type': 'text', 'text': 'Para aprobar cualquier asignatura, el reglamento establece que debes cumplir con un mínimo del **75% de asistencia** en las clases teóricas y el **100% de asistencia** en las clases prácticas obligatorias.\n\n¿Tienes alguna otra duda sobre el reglamento o necesitas ayuda con algún otro trámite académico?', 'index': 0, 'extras': {'signature': 'EnMKcQFpFH0TcX8rQ7jBo2/uwWbTblI6UnNqaoP/24x+vxn/hLrsa7s79Jo0CJGwMgFk+AV+9rgF8FlKfM3wPv/GfZ6uFA3mRD5k/kCfPjiTKJ0osmjqjxOpupURhniyMceoE/Rgl5FTyxKsaGDsC24mJCFd'}}]

> Finished chain.

🤖 Agente responde: [{'type': 'text', 'text': 'Para aprobar cualquier asignatura, el reg